# NYC dinner events from Luma

Luma's official API (`api.lu.ma/public/v1`, used with a `luma-api-...` key) only manages **your own** calendar — it has no city-wide discovery.

Instead we use the public discovery endpoint the luma.com website itself calls (no auth needed):
`GET https://api.lu.ma/discover/get-paginated-events`

It accepts a geographic bounding box (`north`/`south`/`east`/`west`), which returns **every public upcoming event** in that area, paginated via `pagination_cursor`. We pull the whole NYC box (~850 events), then filter for dinners.

In [1]:
import time
import requests

DISCOVER_URL = "https://api.lu.ma/discover/get-paginated-events"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Bounding box covering all five NYC boroughs
NYC_BBOX = {"north": 40.92, "south": 40.49, "west": -74.26, "east": -73.70}


def fetch_all_events(extra_params=None, max_pages=100, delay=0.15):
    """Paginate through the Luma discovery feed and return all event dicts."""
    events, cursor = [], None
    for _ in range(max_pages):
        params = {"pagination_limit": 50, **NYC_BBOX, **(extra_params or {})}
        if cursor:
            params["pagination_cursor"] = cursor
        resp = requests.get(DISCOVER_URL, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        events.extend(entry["event"] for entry in data["entries"])
        if not data.get("has_more"):
            break
        cursor = data["next_cursor"]
        time.sleep(delay)  # be polite to their servers
    return events


all_events = fetch_all_events()
print(f"Fetched {len(all_events)} upcoming public events in NYC")

Fetched 851 upcoming public events in NYC


In [2]:
import pandas as pd

DINNER_KEYWORDS = ("dinner", "supper", "tasting menu", "omakase", "chef's table")


def to_row(e):
    geo = e.get("geo_address_info") or {}
    return {
        "name": e["name"],
        "start": pd.Timestamp(e["start_at"]).tz_convert(e.get("timezone", "America/New_York")),
        "neighborhood": geo.get("sublocality") or geo.get("city"),
        "address": geo.get("full_address"),
        "url": f"https://luma.com/{e['url']}",
    }


dinners = [e for e in all_events if any(k in e["name"].lower() for k in DINNER_KEYWORDS)]
df = pd.DataFrame(to_row(e) for e in dinners).sort_values("start").reset_index(drop=True)

df.to_csv("nyc_dinners.csv", index=False)
print(f"{len(df)} dinner events — saved to nyc_dinners.csv")
df

41 dinner events — saved to nyc_dinners.csv


,name,start,neighborhood,address,url
0,MDS 9 Michelin Dinner July 2026,2026-07-18 17:45:00-04:00,Koreatown,NaN,https://luma.com/MDS9DinnerJuly2026
1,NYC Female Founders Dinner,2026-07-18 18:00:00-04:00,West Village,"Mino Brasserie, 225 W 12th St, New York, NY 10...",https://luma.com/vk8q58sr
2,Max AI Private Practice Advisory Board (Dinner),2026-07-18 18:30:00-04:00,Midtown West,NaN,https://luma.com/s26-adboard-dinner-aad
3,Sunset Supper with Chef Jacob Nguyen,2026-07-19 18:00:00-04:00,Greenwood Heights,"Brooklyn Grange @ Sunset Park, 850 3rd Ave Roo...",https://luma.com/w0eyebmm
4,Women Founders Dinner: Summer Rooftop Series,2026-07-19 18:30:00-04:00,Midtown West,NaN,https://luma.com/connnext-pkqe
5,Chelsea Community Dinner,2026-07-20 18:30:00-04:00,Chelsea,"Neighbor, 176 9th Ave, New York, NY 10011, USA",https://luma.com/lf77sjoe
6,BRIDGES: Executive Summit Welcome Dinner spons...,2026-07-20 19:30:00-04:00,West Village,"L'Artusi Supper Club, 105 Christopher St, New ...",https://luma.com/41f9dg3v
7,AI Founders Supper Club (Hosted by The AI Furn...,2026-07-21 18:00:00-04:00,NoMad,NaN,https://luma.com/xxrgfwv6
8,Supper Club 033,2026-07-21 20:00:00-04:00,Chinatown,NaN,https://luma.com/zw0a0sa9
9,"Solarpunk, Communes, & Co-Living Dinner Series...",2026-07-22 18:00:00-04:00,Fort Greene,NaN,https://luma.com/82mt3j7y


## Guest lists — everything Luma exposes publicly

**Emails are not obtainable.** They exist only in `GET /event/get-guest-list`, which requires being **signed in as the host or a registered guest**. That's private by design — there is no public endpoint, profile field, or query param that returns an attendee's email, so we don't attempt it. (The code below keeps an `email` column, but it stays empty — it's there to make that explicit.)

Everything a guest *chose* to make public **is** available, embedded in each event page's `__NEXT_DATA__` JSON under `featured_guests`. Per guest, the full public field set is:

- **Identity:** name, first/last name, `username`, short bio, avatar image, verified badge, timezone
- **Contact / social:** LinkedIn, Twitter/X, Instagram, TikTok, YouTube, personal website — whichever handles they added
- **Profile meta** (from `luma.com/user/<id>`): join date, #events attended, #events hosted

Caveats:
- `featured_guests` is capped at **10 names** even when more registered, and only appears when the host enabled "show guest list" (17 of our 41 dinners). Hidden lists expose nothing.
- Handles are self-entered, so some are blank, wrong, or junk.

In [3]:
import json
import re

NEXT_DATA_RE = re.compile(
    r'__NEXT_DATA__" type="application/json">(.*?)</script>', re.S
)


def fetch_event_page(slug):
    """Return the embedded event-detail JSON from a public Luma event page."""
    resp = requests.get(f"https://luma.com/{slug}", headers=HEADERS, timeout=30)
    resp.raise_for_status()
    match = NEXT_DATA_RE.search(resp.text)
    return json.loads(match.group(1))["props"]["pageProps"]["initialData"]["data"]


def guest_names(person_list):
    return [p.get("name") or f"{p.get('first_name','')} {p.get('last_name','')}".strip()
            for p in (person_list or [])]


def social_url(kind, handle):
    """Turn a self-entered handle into a full profile URL (best effort)."""
    if not handle:
        return None
    h = str(handle).strip().lstrip("@")
    if h.startswith("http"):
        return h
    if kind == "linkedin":   # stored as a path like "/in/jane" or bare "jane"
        return "https://www.linkedin.com" + (h if h.startswith("/") else "/in/" + h)
    if kind == "twitter":
        return "https://x.com/" + h.lstrip("/")
    if kind == "instagram":
        return "https://instagram.com/" + h.lstrip("/")
    if kind == "tiktok":
        return "https://tiktok.com/@" + h.lstrip("/@")
    if kind == "youtube":
        return "https://youtube.com/@" + h.lstrip("/@")
    if kind == "website":
        return "https://" + h
    return h


def guest_record(g, event_name, event_url):
    """All publicly available fields for one guest."""
    return {
        "event": event_name,
        "event_url": event_url,
        "guest": g.get("name"),
        "first_name": g.get("first_name"),
        "last_name": g.get("last_name"),
        "username": g.get("username"),
        "bio": g.get("bio_short"),
        "verified": g.get("is_verified"),
        "timezone": g.get("timezone"),
        "email": None,  # never exposed publicly by Luma — requires host login
        "linkedin": social_url("linkedin", g.get("linkedin_handle")),
        "twitter": social_url("twitter", g.get("twitter_handle")),
        "instagram": social_url("instagram", g.get("instagram_handle")),
        "tiktok": social_url("tiktok", g.get("tiktok_handle")),
        "youtube": social_url("youtube", g.get("youtube_handle")),
        "website": social_url("website", g.get("website")),
        "avatar_url": g.get("avatar_url"),
        "luma_profile": f"https://luma.com/user/{g['api_id']}" if g.get("api_id") else None,
        "api_id": g.get("api_id"),
    }


guest_rows = []       # one row per (event, guest) — the full public roster
enriched = []         # one row per dinner — counts + host/guest summary

for e in dinners:
    slug = e["url"]
    event_url = f"https://luma.com/{slug}"
    try:
        data = fetch_event_page(slug)
    except Exception as err:
        print(f"skip {slug}: {err}")
        continue

    featured = data.get("featured_guests") or []
    enriched.append({
        "name": e["name"],
        "start": pd.Timestamp(e["start_at"]).tz_convert(e.get("timezone", "America/New_York")),
        "guest_count": data.get("guest_count") or 0,
        "guests_shown": len(featured),          # host reveals at most 10 publicly
        "hosts": ", ".join(guest_names(data.get("hosts"))),
        "featured_guests": ", ".join(guest_names(featured)),
        "url": event_url,
    })

    # hosts are public people too — capture them, tagged as role
    for h in (data.get("hosts") or []):
        rec = guest_record(h, e["name"], event_url)
        rec["role"] = "host"
        guest_rows.append(rec)
    for g in featured:
        rec = guest_record(g, e["name"], event_url)
        rec["role"] = "guest"
        guest_rows.append(rec)

    time.sleep(0.3)  # be polite

dinners_with_guests = pd.DataFrame(enriched).sort_values("start").reset_index(drop=True)
print(f"Enriched {len(dinners_with_guests)} dinners; "
      f"{sum(dinners_with_guests['guests_shown'] > 0)} expose a public guest list")
dinners_with_guests

Enriched 41 dinners; 17 expose a public guest list


,name,start,guest_count,guests_shown,hosts,featured_guests,url
0,MDS 9 Michelin Dinner July 2026,2026-07-18 17:45:00-04:00,0,0,"Million Dollar Sellers (MDS), Eugene Khayman",,https://luma.com/MDS9DinnerJuly2026
1,NYC Female Founders Dinner,2026-07-18 18:00:00-04:00,6,6,X26 - xMcKinsey Entrepreneurs Community,"Alina Vrsaljko, Laura Gallagher, Anna Muszkiew...",https://luma.com/vk8q58sr
2,Max AI Private Practice Advisory Board (Dinner),2026-07-18 18:30:00-04:00,0,0,"Zubair Ahsan, Enea Gjoka, Andrew H Weinstein, ...",,https://luma.com/s26-adboard-dinner-aad
3,Sunset Supper with Chef Jacob Nguyen,2026-07-19 18:00:00-04:00,11,10,Brooklyn Grange,"Michael Benowitz, Jenn Sherman, Katie McGarry,...",https://luma.com/w0eyebmm
4,Women Founders Dinner: Summer Rooftop Series,2026-07-19 18:30:00-04:00,0,0,Celia Davis,,https://luma.com/connnext-pkqe
5,Chelsea Community Dinner,2026-07-20 18:30:00-04:00,5,5,Neighbor,"Mei Tong, Kathleen Ledvora, Franziska Ibscher,...",https://luma.com/lf77sjoe
6,BRIDGES: Executive Summit Welcome Dinner spons...,2026-07-20 19:30:00-04:00,0,0,"Helen Joel, Jennifer Jiao, Neil Daiksel, Katie...",,https://luma.com/41f9dg3v
7,AI Founders Supper Club (Hosted by The AI Furn...,2026-07-21 18:00:00-04:00,0,0,"Angela Mascarenas (The AI Furnace), Hamza Zave...",,https://luma.com/xxrgfwv6
8,Supper Club 033,2026-07-21 20:00:00-04:00,12,10,Nicholas Rhodes,"Megan O'Neill, Jen Toth, Kate Bancroft, Beth F...",https://luma.com/zw0a0sa9
9,"Solarpunk, Communes, & Co-Living Dinner Series...",2026-07-22 18:00:00-04:00,13,10,Chris Morello,"Gillian Morris, Beau, Matt Pfeiffer, Amber, Ja...",https://luma.com/82mt3j7y


In [4]:
# The flattened public roster: one row per (event, person) with every public field
guests = pd.DataFrame(guest_rows)

# Reorder so identity + contact columns lead
cols = ["event", "role", "guest", "first_name", "last_name", "username", "bio",
        "verified", "email", "linkedin", "twitter", "instagram", "tiktok",
        "youtube", "website", "timezone", "avatar_url", "luma_profile",
        "api_id", "event_url"]
guests = guests[[c for c in cols if c in guests.columns]]

dinners_with_guests.to_csv("nyc_dinners_with_guests.csv", index=False)
guests.to_csv("nyc_dinner_guests.csv", index=False)

# How much contact info did we actually get?
has = lambda c: guests[c].notna().sum()
print(f"{len(guests)} people ({(guests['role']=='guest').sum()} guests, "
      f"{(guests['role']=='host').sum()} hosts) across {guests['event'].nunique()} dinners")
print(f"  LinkedIn: {has('linkedin')}   Twitter: {has('twitter')}   "
      f"Instagram: {has('instagram')}   Website: {has('website')}   "
      f"Email: {has('email')} (never public)")
guests.head(20)

227 people (143 guests, 84 hosts) across 40 dinners
  LinkedIn: 130   Twitter: 50   Instagram: 73   Website: 29   Email: 0 (never public)


,event,role,guest,first_name,last_name,username,bio,verified,email,linkedin,twitter,instagram,tiktok,youtube,website,timezone,avatar_url,luma_profile,api_id,event_url
0,MDS 9 Michelin Dinner July 2026,host,Million Dollar Sellers (MDS),Million Dollar Sellers,(MDS),eventsMDS,,False,None,https://www.linkedin.com/company/million-dolla...,https://x.com/mdsonly,https://instagram.com/mdsonly,NaN,https://youtube.com/@milliondollarsellers,https://www.mds.co/,America/Mexico_City,https://images.lumacdn.com/avatars/te/4b2c760c...,https://luma.com/user/usr-4kIUfqrIcO1SwSi,usr-4kIUfqrIcO1SwSi,https://luma.com/MDS9DinnerJuly2026
1,MDS 9 Michelin Dinner July 2026,host,Eugene Khayman,Eugene,Khayman,NaN,,False,None,https://www.linkedin.com/in/million-dollar-sel...,https://x.com/mdsonly,https://instagram.com/mdsonly,NaN,https://youtube.com/@milliondollarsellers,https://mds.co,America/New_York,https://images.lumacdn.com/avatars/g4/af5ed74c...,https://luma.com/user/usr-BoHFbmDK0J9K0sb,usr-BoHFbmDK0J9K0sb,https://luma.com/MDS9DinnerJuly2026
2,NYC Female Founders Dinner,host,X26 - xMcKinsey Entrepreneurs Community,Reece,Griffiths,x26,NaN,False,None,NaN,NaN,NaN,NaN,NaN,NaN,Africa/Johannesburg,https://cdn.lu.ma/avatars/ii/2c86b2c4-5656-445...,https://luma.com/user/usr-FiFJOwIZdOo9BNR,usr-FiFJOwIZdOo9BNR,https://luma.com/vk8q58sr
3,NYC Female Founders Dinner,guest,Alina Vrsaljko,Alina,Vrsaljko,NaN,NaN,False,None,https://www.linkedin.com/in/alina-vrsaljko,https://x.com/Brooklyn,NaN,NaN,NaN,NaN,America/Los_Angeles,https://cdn.lu.ma/avatars-default/avatar_2.png,https://luma.com/user/usr-3S7DQTYHyqfkpMf,usr-3S7DQTYHyqfkpMf,https://luma.com/vk8q58sr
4,NYC Female Founders Dinner,guest,Laura Gallagher,Laura,Gallagher,NaN,NaN,False,None,NaN,NaN,NaN,NaN,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_15.png,https://luma.com/user/usr-7Pm8Kl29goo6fVm,usr-7Pm8Kl29goo6fVm,https://luma.com/vk8q58sr
5,NYC Female Founders Dinner,guest,Anna Muszkiewicz,NaN,NaN,NaN,NaN,False,None,https://www.linkedin.com/in/anna-muszkiewicz-p...,NaN,NaN,NaN,NaN,NaN,Asia/Kolkata,https://cdn.lu.ma/avatars-default/avatar_15.png,https://luma.com/user/usr-B0Nat0CpoWP0lYq,usr-B0Nat0CpoWP0lYq,https://luma.com/vk8q58sr
6,NYC Female Founders Dinner,guest,Tina Yuan,Tina,Yuan,NaN,NaN,False,None,https://www.linkedin.com/in/tinayuan8,NaN,NaN,NaN,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_22.png,https://luma.com/user/usr-eOJpHQq5NaQxAdO,usr-eOJpHQq5NaQxAdO,https://luma.com/vk8q58sr
7,NYC Female Founders Dinner,guest,Sophie Jewsbury,Sophie,Jewsbury,NaN,,False,None,https://www.linkedin.com/in/sophiejewsbury,NaN,NaN,NaN,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_2.png,https://luma.com/user/usr-FlwPN5XlJRX6Q0t,usr-FlwPN5XlJRX6Q0t,https://luma.com/vk8q58sr
8,NYC Female Founders Dinner,guest,Iness Arabi,Iness,Arabi,NaN,NaN,False,None,NaN,NaN,NaN,NaN,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_42.png,https://luma.com/user/usr-xiYVy9eYrRwz5cV,usr-xiYVy9eYrRwz5cV,https://luma.com/vk8q58sr
9,Max AI Private Practice Advisory Board (Dinner),host,Zubair Ahsan,Zubair,Ahsan,zubairahsan,,False,None,https://www.linkedin.com/in/zubairahsan,NaN,https://instagram.com/zubairahsan,NaN,NaN,NaN,America/New_York,https://images.lumacdn.com/avatars/pr/6e0ca7d1...,https://luma.com/user/usr-YLVi5IiVl0Zxaq1,usr-YLVi5IiVl0Zxaq1,https://luma.com/s26-adboard-dinner-aad


In [5]:
# Optional: enrich each unique person with profile-page meta
# (join date, #events attended, #events hosted). One extra request per person.
FETCH_PROFILE_META = True


def fetch_profile(api_id):
    resp = requests.get(f"https://luma.com/user/{api_id}", headers=HEADERS, timeout=30)
    resp.raise_for_status()
    match = NEXT_DATA_RE.search(resp.text)
    return json.loads(match.group(1))["props"]["pageProps"]["initialData"]


if FETCH_PROFILE_META:
    meta = {}
    ids = [a for a in guests["api_id"].dropna().unique()]
    for i, api_id in enumerate(ids, 1):
        try:
            d = fetch_profile(api_id)
            meta[api_id] = {
                "joined_at": d.get("joined_at"),
                "events_attended": d.get("event_attended_count"),
                "events_hosted": d.get("event_hosted_count"),
            }
        except Exception as err:
            print(f"skip {api_id}: {err}")
        time.sleep(0.25)
    meta_df = pd.DataFrame.from_dict(meta, orient="index")
    guests = guests.merge(meta_df, left_on="api_id", right_index=True, how="left")
    guests.to_csv("nyc_dinner_guests.csv", index=False)
    print(f"Added profile meta for {len(meta)} unique people")

guests.head(20)

Added profile meta for 214 unique people


,event,role,guest,first_name,last_name,username,bio,verified,email,linkedin,...,youtube,website,timezone,avatar_url,luma_profile,api_id,event_url,joined_at,events_attended,events_hosted
0,MDS 9 Michelin Dinner July 2026,host,Million Dollar Sellers (MDS),Million Dollar Sellers,(MDS),eventsMDS,,False,None,https://www.linkedin.com/company/million-dolla...,...,https://youtube.com/@milliondollarsellers,https://www.mds.co/,America/Mexico_City,https://images.lumacdn.com/avatars/te/4b2c760c...,https://luma.com/user/usr-4kIUfqrIcO1SwSi,usr-4kIUfqrIcO1SwSi,https://luma.com/MDS9DinnerJuly2026,2025-05-07T22:45:22.607Z,3,128
1,MDS 9 Michelin Dinner July 2026,host,Eugene Khayman,Eugene,Khayman,NaN,,False,None,https://www.linkedin.com/in/million-dollar-sel...,...,https://youtube.com/@milliondollarsellers,https://mds.co,America/New_York,https://images.lumacdn.com/avatars/g4/af5ed74c...,https://luma.com/user/usr-BoHFbmDK0J9K0sb,usr-BoHFbmDK0J9K0sb,https://luma.com/MDS9DinnerJuly2026,2025-04-01T13:45:19.145Z,30,53
2,NYC Female Founders Dinner,host,X26 - xMcKinsey Entrepreneurs Community,Reece,Griffiths,x26,NaN,False,None,NaN,...,NaN,NaN,Africa/Johannesburg,https://cdn.lu.ma/avatars/ii/2c86b2c4-5656-445...,https://luma.com/user/usr-FiFJOwIZdOo9BNR,usr-FiFJOwIZdOo9BNR,https://luma.com/vk8q58sr,2023-01-11T12:05:37.578Z,3,41
3,NYC Female Founders Dinner,guest,Alina Vrsaljko,Alina,Vrsaljko,NaN,NaN,False,None,https://www.linkedin.com/in/alina-vrsaljko,...,NaN,NaN,America/Los_Angeles,https://cdn.lu.ma/avatars-default/avatar_2.png,https://luma.com/user/usr-3S7DQTYHyqfkpMf,usr-3S7DQTYHyqfkpMf,https://luma.com/vk8q58sr,2025-10-15T14:07:24.607Z,169,2
4,NYC Female Founders Dinner,guest,Laura Gallagher,Laura,Gallagher,NaN,NaN,False,None,NaN,...,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_15.png,https://luma.com/user/usr-7Pm8Kl29goo6fVm,usr-7Pm8Kl29goo6fVm,https://luma.com/vk8q58sr,2026-03-28T17:35:01.028Z,3,0
5,NYC Female Founders Dinner,guest,Anna Muszkiewicz,NaN,NaN,NaN,NaN,False,None,https://www.linkedin.com/in/anna-muszkiewicz-p...,...,NaN,NaN,Asia/Kolkata,https://cdn.lu.ma/avatars-default/avatar_15.png,https://luma.com/user/usr-B0Nat0CpoWP0lYq,usr-B0Nat0CpoWP0lYq,https://luma.com/vk8q58sr,2024-09-12T01:20:29.694Z,4,0
6,NYC Female Founders Dinner,guest,Tina Yuan,Tina,Yuan,NaN,NaN,False,None,https://www.linkedin.com/in/tinayuan8,...,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_22.png,https://luma.com/user/usr-eOJpHQq5NaQxAdO,usr-eOJpHQq5NaQxAdO,https://luma.com/vk8q58sr,2025-09-24T16:37:19.936Z,12,0
7,NYC Female Founders Dinner,guest,Sophie Jewsbury,Sophie,Jewsbury,NaN,,False,None,https://www.linkedin.com/in/sophiejewsbury,...,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_2.png,https://luma.com/user/usr-FlwPN5XlJRX6Q0t,usr-FlwPN5XlJRX6Q0t,https://luma.com/vk8q58sr,2024-09-06T22:44:44.184Z,81,1
8,NYC Female Founders Dinner,guest,Iness Arabi,Iness,Arabi,NaN,NaN,False,None,NaN,...,NaN,NaN,America/New_York,https://cdn.lu.ma/avatars-default/avatar_42.png,https://luma.com/user/usr-xiYVy9eYrRwz5cV,usr-xiYVy9eYrRwz5cV,https://luma.com/vk8q58sr,2026-06-09T13:47:04.101Z,4,0
9,Max AI Private Practice Advisory Board (Dinner),host,Zubair Ahsan,Zubair,Ahsan,zubairahsan,,False,None,https://www.linkedin.com/in/zubairahsan,...,NaN,NaN,America/New_York,https://images.lumacdn.com/avatars/pr/6e0ca7d1...,https://luma.com/user/usr-YLVi5IiVl0Zxaq1,usr-YLVi5IiVl0Zxaq1,https://luma.com/s26-adboard-dinner-aad,2020-10-20T05:09:05.559Z,8,14
